# Pepper Inference

Colab-friendly notebook for loading a Textual Inversion or DreamBooth
checkpoint produced by this repo and sampling Pepper images from prompts
with the manual DDIM sampler defined in `src/utils/sampling.py`.

**Important**: this notebook only *uses* the building blocks in `src/`; it
does not introduce a pipeline wrapper. Every diffusion step is still run
through our hand-written sampler.

## 0. Colab bootstrap (skip locally)

In [ ]:
from pathlib import Path
import subprocess

RUNNING_IN_COLAB = False
try:
    import google.colab  # noqa: F401
    RUNNING_IN_COLAB = True
except ImportError:
    pass

if RUNNING_IN_COLAB:
    from google.colab import drive

    DRIVE_MOUNT_ROOT = '/content/gdrive'
    drive.mount(DRIVE_MOUNT_ROOT)

    MY_DRIVE_ROOT = Path(DRIVE_MOUNT_ROOT) / 'MyDrive'
    DRIVE_PROJECT_ROOT = MY_DRIVE_ROOT / 'persistent-pepper'
    DRIVE_CREDENTIALS_DIRECTORY = MY_DRIVE_ROOT / 'cs5788'
    DRIVE_CACHE_ROOT = DRIVE_PROJECT_ROOT / 'cache'
    DRIVE_CHECKPOINTS_ROOT = DRIVE_CACHE_ROOT / 'checkpoints'
    DRIVE_PROCESSED_IMAGES_DIRECTORY = DRIVE_CACHE_ROOT / 'processed'
    DRIVE_SD_MODEL_DIRECTORY = DRIVE_CACHE_ROOT / 'sd-v1-5'

    GITHUB_TOKEN_FILE_PATH = DRIVE_CREDENTIALS_DIRECTORY / 'token.txt'
    GITHUB_USERNAME_FILE_PATH = DRIVE_CREDENTIALS_DIRECTORY / 'git_username.txt'
    GITHUB_REPOSITORY_OWNER = 'AaronGoldblatt'
    GITHUB_REPOSITORY_NAME = 'persistent-pepper'
    BRANCH_NAME = 'main'
    LOCAL_REPOSITORY_ROOT = Path('/content/persistent-pepper')

    if not GITHUB_TOKEN_FILE_PATH.is_file():
        raise FileNotFoundError(
            f'Missing GitHub token file: {GITHUB_TOKEN_FILE_PATH}. '
            'Create it in Drive with only the token on the first line.'
        )
    if not GITHUB_USERNAME_FILE_PATH.is_file():
        raise FileNotFoundError(
            f'Missing GitHub username file: {GITHUB_USERNAME_FILE_PATH}. '
            'Create it in Drive with only your GitHub username on the first line.'
        )

    github_token = GITHUB_TOKEN_FILE_PATH.read_text().strip().splitlines()[0]
    github_username = GITHUB_USERNAME_FILE_PATH.read_text().strip().splitlines()[0]
    authenticated_repository_url = (
        f'https://{github_token}@github.com/'
        f'{GITHUB_REPOSITORY_OWNER}/{GITHUB_REPOSITORY_NAME}.git'
    )
    public_repository_url = (
        f'https://github.com/{GITHUB_REPOSITORY_OWNER}/{GITHUB_REPOSITORY_NAME}.git'
    )

    def run_git_command(command_parts, working_directory=None):
        completed_process = subprocess.run(
            command_parts,
            cwd=working_directory,
            capture_output=True,
            text=True,
        )
        if completed_process.returncode != 0:
            sanitized_command_parts = [
                command_part.replace(github_token, '[GITHUB_TOKEN]')
                for command_part in command_parts
            ]
            sanitized_stderr = completed_process.stderr.replace(github_token, '[GITHUB_TOKEN]')
            sanitized_stdout = completed_process.stdout.replace(github_token, '[GITHUB_TOKEN]')
            raise RuntimeError(
                'Git command failed:\n'
                f'command: {sanitized_command_parts}\n'
                f'stdout: {sanitized_stdout}\n'
                f'stderr: {sanitized_stderr}'
            )
        return completed_process

    if not LOCAL_REPOSITORY_ROOT.exists():
        run_git_command(
            [
                'git',
                'clone',
                '--branch',
                BRANCH_NAME,
                '--single-branch',
                authenticated_repository_url,
                str(LOCAL_REPOSITORY_ROOT),
            ]
        )
    else:
        run_git_command(
            ['git', 'remote', 'set-url', 'origin', authenticated_repository_url],
            working_directory=LOCAL_REPOSITORY_ROOT,
        )
        run_git_command(
            ['git', 'checkout', BRANCH_NAME],
            working_directory=LOCAL_REPOSITORY_ROOT,
        )
        run_git_command(
            ['git', 'pull', '--ff-only'],
            working_directory=LOCAL_REPOSITORY_ROOT,
        )

    run_git_command(
        ['git', 'remote', 'set-url', 'origin', public_repository_url],
        working_directory=LOCAL_REPOSITORY_ROOT,
    )

    %cd {LOCAL_REPOSITORY_ROOT}
    !python -m pip -q install -r requirements.txt

    REPOSITORY_ROOT = str(LOCAL_REPOSITORY_ROOT)
    print(f'Using private repo {GITHUB_REPOSITORY_OWNER}/{GITHUB_REPOSITORY_NAME} as {github_username}.')
else:
    REPOSITORY_ROOT = str(Path().resolve().parent)
    DRIVE_CHECKPOINTS_ROOT = Path(REPOSITORY_ROOT) / 'checkpoints'
    DRIVE_PROCESSED_IMAGES_DIRECTORY = Path(REPOSITORY_ROOT) / 'data' / 'processed'
    DRIVE_SD_MODEL_DIRECTORY = Path(REPOSITORY_ROOT) / 'checkpoints' / 'sd-v1-5'

print('Repository root:', REPOSITORY_ROOT)
print('Checkpoints root:', DRIVE_CHECKPOINTS_ROOT)

## 1. Put the repo on `sys.path` and import building blocks

In [ ]:
import sys
sys.path.insert(0, REPOSITORY_ROOT)

import torch
from pathlib import Path
import matplotlib.pyplot as plt

from src.models import load_stable_diffusion_components
from src.utils.sampling import SamplingConfiguration, sample_images_from_prompts

pretrained_model_name_or_path = str(DRIVE_SD_MODEL_DIRECTORY)
if not Path(pretrained_model_name_or_path).exists():
    pretrained_model_name_or_path = 'stable-diffusion-v1-5/stable-diffusion-v1-5'

stable_diffusion_components = load_stable_diffusion_components(
    pretrained_model_name_or_path=pretrained_model_name_or_path,
    working_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
)
print('Loaded SD 1.5 onto', stable_diffusion_components.working_device)
print('Using model source:', pretrained_model_name_or_path)

## 2a. Load a Textual Inversion checkpoint (skip if using DreamBooth)

In [ ]:
from scripts.evaluate import load_textual_inversion_embedding_into_components

learned_embedding_checkpoint_path = (
    DRIVE_CHECKPOINTS_ROOT / 'textual_inversion' / 'learned_embedding_final.bin'
)

placeholder_token_string = load_textual_inversion_embedding_into_components(
    stable_diffusion_components=stable_diffusion_components,
    learned_embedding_checkpoint_path=learned_embedding_checkpoint_path,
)
print('Loaded textual inversion embedding:', placeholder_token_string)
print('Checkpoint path:', learned_embedding_checkpoint_path)

subject_phrase_for_prompts = placeholder_token_string

## 2b. Load a DreamBooth checkpoint (skip if using Textual Inversion)

In [ ]:
from scripts.evaluate import load_dreambooth_weights_into_components

dreambooth_checkpoint_directory = DRIVE_CHECKPOINTS_ROOT / 'dreambooth' / 'latest'

load_dreambooth_weights_into_components(
    stable_diffusion_components=stable_diffusion_components,
    dreambooth_checkpoint_directory=dreambooth_checkpoint_directory,
)
print('Loaded DreamBooth checkpoint:', dreambooth_checkpoint_directory)

subject_phrase_for_prompts = 'sks dog'

## 3. Sample images

In [ ]:
prompt_templates = [
    'a photo of {subject} wearing a red wool sweater',
    'an oil painting of {subject} in the style of Van Gogh',
    'a photo of {subject} on the surface of the moon',
    'a pencil sketch of {subject}',
]
fully_substituted_prompts = [
    prompt_template.format(subject=subject_phrase_for_prompts)
    for prompt_template in prompt_templates
]

sampling_configuration = SamplingConfiguration(
    number_of_inference_steps=50,
    classifier_free_guidance_scale=7.5,
    random_seed=42,
)

generated_pil_images = sample_images_from_prompts(
    stable_diffusion_components=stable_diffusion_components,
    prompt_texts=fully_substituted_prompts,
    sampling_configuration=sampling_configuration,
)

figure, axes_grid = plt.subplots(
    1,
    len(generated_pil_images),
    figsize=(4 * len(generated_pil_images), 4),
)
if len(generated_pil_images) == 1:
    axes_grid = [axes_grid]
for axis, generated_pil_image, fully_substituted_prompt in zip(
    axes_grid,
    generated_pil_images,
    fully_substituted_prompts,
):
    axis.imshow(generated_pil_image)
    axis.set_title(fully_substituted_prompt, fontsize=8)
    axis.axis('off')
plt.tight_layout()
plt.show()

## 4. Optional: CLIP-based identity / prompt alignment report

In [ ]:
from src.utils.metrics import (
    compute_clip_similarity_report,
    format_report_as_human_readable_text,
    load_pil_images_from_directory,
)

reference_pil_images = load_pil_images_from_directory(
    DRIVE_PROCESSED_IMAGES_DIRECTORY
)
clip_similarity_report = compute_clip_similarity_report(
    generated_pil_images=generated_pil_images,
    reference_pil_images=reference_pil_images,
    prompts_used_for_generation=fully_substituted_prompts,
)
print(format_report_as_human_readable_text(clip_similarity_report))

## 5. Custom prompt: one method or both

Use this section when you want to type one prompt and render it with Textual Inversion, DreamBooth, or both methods side-by-side. The prompt should contain `{subject}`, which will be replaced with either `<pepper>` or `sks dog` depending on the method.

This cell reloads a fresh SD 1.5 component bundle per method so the two methods do not accidentally overwrite each other's weights.

In [ ]:
import gc
import textwrap

from scripts.evaluate import (
    load_dreambooth_weights_into_components,
    load_textual_inversion_embedding_into_components,
)

# Edit these two values for interactive use.
CUSTOM_PROMPT_TEMPLATE = 'a cinematic photo of {subject} wearing sunglasses on a beach at sunset'
METHODS_TO_RENDER = ['textual_inversion', 'dreambooth']  # choose: ['textual_inversion'], ['dreambooth'], or both

CUSTOM_SAMPLING_CONFIGURATION = SamplingConfiguration(
    number_of_inference_steps=50,
    classifier_free_guidance_scale=7.5,
    random_seed=123,
)

CUSTOM_METHOD_SETTINGS = {
    'textual_inversion': {
        'display_name': 'Textual Inversion (<pepper>)',
        'subject_identifier_phrase': '<pepper>',
        'checkpoint_path': DRIVE_CHECKPOINTS_ROOT / 'textual_inversion' / 'learned_embedding_final.bin',
    },
    'dreambooth': {
        'display_name': 'DreamBooth (sks dog)',
        'subject_identifier_phrase': 'sks dog',
        'checkpoint_path': DRIVE_CHECKPOINTS_ROOT / 'dreambooth' / 'latest',
    },
}

custom_generated_images_by_method = {}
custom_prompt_by_method = {}

for method_name in METHODS_TO_RENDER:
    if method_name not in CUSTOM_METHOD_SETTINGS:
        raise ValueError(
            f'Unknown method {method_name!r}. Expected one of: {list(CUSTOM_METHOD_SETTINGS)}'
        )

    method_settings = CUSTOM_METHOD_SETTINGS[method_name]
    custom_prompt_text = CUSTOM_PROMPT_TEMPLATE.format(
        subject=method_settings['subject_identifier_phrase']
    )
    custom_prompt_by_method[method_name] = custom_prompt_text

    print(f"Generating with {method_settings['display_name']}...")
    method_components = load_stable_diffusion_components(
        pretrained_model_name_or_path=pretrained_model_name_or_path,
        working_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    )

    if method_name == 'textual_inversion':
        load_textual_inversion_embedding_into_components(
            stable_diffusion_components=method_components,
            learned_embedding_checkpoint_path=method_settings['checkpoint_path'],
        )
    elif method_name == 'dreambooth':
        load_dreambooth_weights_into_components(
            stable_diffusion_components=method_components,
            dreambooth_checkpoint_directory=method_settings['checkpoint_path'],
        )

    generated_image = sample_images_from_prompts(
        stable_diffusion_components=method_components,
        prompt_texts=[custom_prompt_text],
        sampling_configuration=CUSTOM_SAMPLING_CONFIGURATION,
    )[0]
    custom_generated_images_by_method[method_name] = generated_image

    del method_components
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

figure, axes_grid = plt.subplots(
    1,
    len(METHODS_TO_RENDER),
    figsize=(5.0 * len(METHODS_TO_RENDER), 5.2),
    squeeze=False,
)

for axis, method_name in zip(axes_grid[0], METHODS_TO_RENDER):
    method_settings = CUSTOM_METHOD_SETTINGS[method_name]
    axis.imshow(custom_generated_images_by_method[method_name])
    axis.set_title(
        method_settings['display_name'],
        fontsize=13,
        weight='bold',
    )
    axis.set_xlabel(
        textwrap.fill(custom_prompt_by_method[method_name], width=45),
        fontsize=9,
    )
    axis.set_xticks([])
    axis.set_yticks([])

figure.suptitle(
    'Custom Prompt Result',
    fontsize=17,
    weight='bold',
    y=1.03,
)
figure.tight_layout()
plt.show()

custom_output_directory = DRIVE_CHECKPOINTS_ROOT / 'custom_prompt_outputs'
custom_output_directory.mkdir(parents=True, exist_ok=True)

for method_name, generated_image in custom_generated_images_by_method.items():
    safe_prompt_stub = ''.join(
        character if character.isalnum() else '_'
        for character in CUSTOM_PROMPT_TEMPLATE.format(subject='pepper').lower()
    )[:80]
    output_image_path = custom_output_directory / f'{method_name}_{safe_prompt_stub}.png'
    generated_image.save(output_image_path)
    print(f'Saved {method_name} image to:', output_image_path)